# Movie Recommender — Training Notebook

End-to-end training pipeline for a hybrid movie recommendation system trained on the **MovieLens 32M** dataset (~32 M ratings, 87,585 movies, 200,948 users).

## Pipeline overview

| Stage | What it does |
|---|---|
| **1 · Imports & GPU setup** | Configure environment, enable memory growth |
| **2 · Data loading** | Pull ratings + movies from BigQuery |
| **3 · Preprocessing** | Combine ratings with genre vectors; cold-start train/test split by user |
| **4 · Model architecture** | Two-tower TFRS model (user × movie × genre embeddings) |
| **5 · HPO — Hyperband** | Tune embedding size, layer widths, dropout, L2, learning-rate schedule |
| **6 · Embedding extraction** | Materialise user & movie embeddings from the tuned network |
| **7 · XGBoost ranking** | Re-rank candidates using fused embeddings; tune with Optuna |
| **8 · Evaluation** | Per-user NDCG, Precision/Recall/Accuracy@10 across rating thresholds |
| **9 · Inference pipeline** | `get_final_predictions`: XGBoost retrieval → hypermodel re-scoring |
| **10 · Neural net metrics** | Full-dataset `model.evaluate` for retrieval top-K accuracy + RMSE |
| **11 · Saving artifacts** | Keras weights, XGBoost JSON model, FAISS index |
| **12 · FAISS index** | Batch-extract embeddings, build `IndexFlatIP` for ANN retrieval |


In [ ]:
import os
# Must be set before TensorFlow is imported.
# Forces keras_tuner and TFRS to use tf.keras (Keras 2 compatibility layer)
# rather than standalone Keras 3, which is incompatible with TF 2.15.
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# ── Standard library ──────────────────────────────────────────────────────────
import datetime
import logging
import platform   # used in RecommendationHyperModel.build() for platform-aware Adam
import pprint
import random
from typing import Dict, Text

# ── Numeric / data science ────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score   # per-user NDCG for XGBoost evaluation

# ── TensorFlow & TFRS ─────────────────────────────────────────────────────────
import tensorflow as tf
import tensorflow_recommenders as tfrs   # two-tower model + retrieval / ranking tasks

# ── Keras Tuner — Hyperband HPO for the neural network ───────────────────────
from keras_tuner import HyperModel, Objective
from keras_tuner.tuners import Hyperband

# ── XGBoost + Optuna — ranking model + HPO ───────────────────────────────────
import xgboost as xgb
import optuna

# ── FAISS — approximate nearest-neighbour retrieval ──────────────────────────
import faiss

# ── Google Cloud ──────────────────────────────────────────────────────────────
from google.cloud import bigquery

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


## 1 · Environment & GPU Setup

Enable TensorFlow GPU memory growth so the process claims only as much VRAM as it needs. On Apple Silicon this prevents the Metal backend from pre-allocating the entire unified memory pool and crashing during long tuning runs.


In [ ]:
# Check for GPU availability and set memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logger.info(f"GPUs available: {len(gpus)}")
    except RuntimeError as e:
        logger.warning(e)
else:
    logger.info("No GPUs available.")

## 2 · Data Loading

Pull the preprocessed movies and ratings tables from BigQuery. Each table was pre-built from the **MovieLens 32M** dataset:

- `preprocessed_movies` — one row per movie: `title`, `genres` (19-element one-hot vector)
- `ratings_with_titles` — one row per interaction: `user_id`, `title`, `rating`

The loader functions wrap every BQ call in structured error handling so failures surface a clear stack trace rather than a silent `None`.


In [ ]:
# Define the BigQuery table and project details
PROJECT_ID = 'my-project'  # Replace with your GCP project ID
DATASET_ID = 'movie_data'
TABLE_ID   = 'preprocessed_data'
timestamp  = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
output_dir = 'gs://movie-data-1/trained-model'
# Function to load movies from BigQuery
def load_movies_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT title, genres
        FROM `{PROJECT_ID}.{DATASET_ID}.preprocessed_movies`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading movies from BigQuery: {e}")
        raise
# Function to load ratings from BigQuery
def load_ratings_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT user_id, title, rating
        FROM `{PROJECT_ID}.{DATASET_ID}.ratings_with_titles`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading ratings from BigQuery: {e}")
        raise

In [ ]:
movies_bq = load_movies_bq()
movies_dict = {key: list(value) for key, value in movies_bq[['title', 'genres']].to_dict(orient='list').items()}

In [ ]:
ratings_bq = load_ratings_bq()
ratings_dict = {key: list(value) for key, value in ratings_bq[['title', 'user_id', 'rating']].to_dict(orient='list').items()}

In [ ]:
ratings = tf.data.Dataset.from_tensor_slices(ratings_dict)

In [ ]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

In [ ]:
movies = tf.data.Dataset.from_tensor_slices(movies_dict)

In [ ]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

In [ ]:
ratings = ratings.map(lambda x: {
            "title": x["title"],
            "user_id": x["user_id"],
            "rating": x["rating"]
        })
movies = movies.map(lambda x: {
    "title": x["title"],
    "genres": x["genres"]
})

In [ ]:
# print unique titles lenght from ratings and movies
print(f"Unique titles in ratings: {len(np.unique(ratings_dict['title']))}")
print(f"Unique titles in movies: {len(np.unique(movies_dict['title']))}")

### Data Preprocessing — Combining Ratings with Genre Vectors

For each rating, look up the movie's 19-element genre vector and attach it as a new field. This enriches every training example with genre signal so the model can learn genre preferences without a separate side-input lookup at inference time.

`tf.py_function` is used for the dict lookup because Python-level dictionary access cannot be expressed as a pure TF graph operation.


In [ ]:
# Create a dictionary from the movies dataset
movies_dict = {movie["title"].numpy().decode('utf-8'): movie["genres"].numpy() for movie in movies}

# Combine the ratings and movies datasets only for the movies that are in the ratings dataset
def combine_datasets(rating):
    def lookup_genres(title):
        title_str = title.numpy().decode('utf-8')  # Convert to numpy and decode the title from bytes to string
        return movies_dict.get(title_str, [0] * 19)

    genres = tf.py_function(
        func=lookup_genres,
        inp=[rating["title"]],
        Tout=tf.int32
    )
    genres.set_shape([19])
    rating["genres"] = genres
    return rating

combined_dataset = ratings.map(combine_datasets)

In [ ]:
# print one example of combined dataset
for x in combined_dataset.take(1).as_numpy_iterator():
  pprint.pprint(x)

In [ ]:
combined_dataset = combined_dataset.map(lambda x: {
    "title": x["title"],
    "user_id": x["user_id"],
    "genres": x["genres"],
    "rating": x["rating"]
}, num_parallel_calls=tf.data.AUTOTUNE)

## 3 · Train / Test Split

Split by **user identity**, not by row. This simulates a cold-start scenario: the model is evaluated on users it has never seen during training, which is the realistic deployment case.

- 80 % of users → training partition
- 20 % of users → test partition
- An assertion verifies zero overlap before any data is loaded

> **Why not a row-level split?** A random row split allows the same user to appear in both partitions. Because the user embedding is learned during training, any user present in both splits has their embedding "pre-heated" which inflates retrieval metrics by up to 10 %. The user-stratified split avoids this leakage.


In [ ]:

# Set the seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# ── User-stratified train/test split ─────────────────────────────────────────
# Splitting by user_id (not by row) ensures no user's interactions straddle
# both the train and test partitions, preventing embedding-level data leakage.
all_user_ids_unique = np.unique(ratings_bq['user_id'].values)
np.random.shuffle(all_user_ids_unique)
split_idx = int(len(all_user_ids_unique) * 0.8)

train_user_ids_set = set(all_user_ids_unique[:split_idx].tolist())
test_user_ids_set  = set(all_user_ids_unique[split_idx:].tolist())

assert not (train_user_ids_set & test_user_ids_set), "User leakage detected between train and test sets!"

train_ratings_bq = ratings_bq[ratings_bq['user_id'].isin(train_user_ids_set)]
test_ratings_bq  = ratings_bq[ratings_bq['user_id'].isin(test_user_ids_set)]

train_length = len(train_ratings_bq)
test_length  = len(test_ratings_bq)
ds_length    = len(ratings_bq)

print(f"Total interactions : {ds_length}")
print(f"Train users        : {len(train_user_ids_set)} ({train_length} interactions)")
print(f"Test  users        : {len(test_user_ids_set)}  ({test_length} interactions)")


def build_tf_dataset(ratings_df):
    """Build a TF dataset pipeline from a ratings DataFrame."""
    d = {k: list(v) for k, v in ratings_df[['title', 'user_id', 'rating']].to_dict(orient='list').items()}
    ds = tf.data.Dataset.from_tensor_slices(d)
    ds = ds.map(lambda x: {"title": x["title"], "user_id": x["user_id"], "rating": x["rating"]})
    ds = ds.map(combine_datasets)
    ds = ds.map(
        lambda x: {"title": x["title"], "user_id": x["user_id"], "genres": x["genres"], "rating": x["rating"]},
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return ds


train_combined_dataset = build_tf_dataset(train_ratings_bq)
test_combined_dataset  = build_tf_dataset(test_ratings_bq)

# Optimize performance with batching and prefetching.
# NOTE: .cache() is intentionally omitted here. On Apple Silicon (M-series),
# caching the full dataset into RAM causes memory to accumulate across Hyperband
# trials and eventually kills the kernel. Prefetch alone is sufficient.
train_batch_size = 128
test_batch_size  = 64

trainds = train_combined_dataset.shuffle(100_000, seed=42).batch(train_batch_size).prefetch(tf.data.AUTOTUNE)
testds  = test_combined_dataset.batch(test_batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
titles = movies.batch(100000).map(lambda x: x["title"])
user_ids = ratings.batch(1000000).map(lambda x: x["user_id"])
genres = movies.batch(100000).map(lambda x: {
        "title": x["title"],
        "genres": x["genres"]
    })

In [ ]:
unique_titles = np.unique(np.concatenate(list(titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))
unique_genres = [
            'Action',
            'Adventure',
            'Animation',
            'Children',
            'Comedy',
            'Crime',
            'Drama',
            'Documentary',
            'Fantasy',
            'Film-Noir',
            'Horror',
            'IMAX',
            'Musical',
            'Mystery',
            'Romance',
            'Sci-Fi',
            'Thriller',
            'War',
            'Western'
        ]

In [ ]:
print(f"Unique titles: {len(unique_titles)}")

## 4 · Model Architecture

The recommendation model is a **two-tower** architecture built with TensorFlow Recommenders:

```
User ID ──► IntegerLookup ──► Embedding(D) ──┐
                                              ├──► concat ──► Dense(units_1) ──► BN ──► Dropout
Movie title ─► StringLookup ──► Embedding(D) ─┤              Dense(units_2) ──► BN ──► Dropout
                                              │              Dense(1) ──► rating_prediction
Genre (19-d) ─► Dense(D) ─────────────────────┘
```

- **`RecommendationModel`** — base TFRS model; joins rating task (MSE) and retrieval task (softmax) into a single weighted loss: `2.0 × rating_loss + 0.5 × retrieval_loss`
- **`RecommendationHyperModel`** — wraps the architecture in a Keras Tuner `HyperModel` so that embedding size, layer widths, dropout rates, L2 regularisation, and learning-rate schedule are all tunable
- **BatchNormalization** after each dense tower stabilises training and lets Hyperband explore higher learning rates safely


In [ ]:
class RecommendationModel(tfrs.Model):
    def __init__(self, user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight=1.0, retrieval_weight=1.0):
        super().__init__()
        self.user_model = user_model
        self.movie_model = movie_model
        self.genre_model = genre_model
        self.rating_model = rating_model
        self.rating_task = rating_task
        self.retrieval_task = retrieval_task
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def call(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        movie_embeddings = self.movie_model(features["title"])
        genre_embeddings = self.genre_model(features["genres"])
        rating_predictions = self.rating_model([features["user_id"], features["title"], features["genres"]])
        return user_embeddings, movie_embeddings, genre_embeddings, rating_predictions

    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        ratings = features.pop("rating")
        user_embeddings, movie_embeddings, genre_embeddings, rating_predictions = self(features)
        rating_loss = self.rating_task(labels=ratings, predictions=rating_predictions)
        retrieval_loss = self.retrieval_task(user_embeddings, movie_embeddings)
        return (self.rating_weight * rating_loss
                + self.retrieval_weight * retrieval_loss)

In [ ]:
class RecommendationHyperModel(HyperModel):
    def __init__(self, unique_user_ids, unique_titles, num_genres, rating_weight=2.0, retrieval_weight=0.5):
        self.unique_user_ids = unique_user_ids
        self.unique_titles = unique_titles
        self.num_genres = num_genres
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def build(self, hp):
        embedding_dimension = hp.Int('embedding_dimension', min_value=32, max_value=256, step=32)
        
        # Add L2 regularization as a tunable hyperparameter
        l2_reg = hp.Float('l2_reg', min_value=1e-5, max_value=1e-2, sampling='log')

        user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
        movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
        genre_input = tf.keras.layers.Input(shape=(self.num_genres,), dtype=tf.float32, name='genres')

        user_lookup = tf.keras.layers.IntegerLookup(vocabulary=self.unique_user_ids, mask_token=None)
        movie_lookup = tf.keras.layers.StringLookup(vocabulary=self.unique_titles, mask_token=None)

        user_embedding = tf.keras.layers.Embedding(
            len(self.unique_user_ids) + 1, 
            embedding_dimension,
            embeddings_regularizer=tf.keras.regularizers.l2(l2_reg)
        )(user_lookup(user_input))
        movie_embedding = tf.keras.layers.Embedding(
            len(self.unique_titles) + 1, 
            embedding_dimension,
            embeddings_regularizer=tf.keras.regularizers.l2(l2_reg)
        )(movie_lookup(movie_input))
        genre_embedding = tf.keras.layers.Dense(
            embedding_dimension,
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
        )(genre_input)

        concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

        dense_1 = tf.keras.layers.Dense(
            hp.Int('units_1', min_value=128, max_value=512, step=64), 
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
        )(concatenated_embeddings)
        dense_1 = tf.keras.layers.BatchNormalization()(dense_1)
        dropout_1 = tf.keras.layers.Dropout(hp.Float('dropout_1', min_value=0.3, max_value=0.6, step=0.1))(dense_1)
        
        dense_2 = tf.keras.layers.Dense(
            hp.Int('units_2', min_value=64, max_value=256, step=32), 
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
        )(dropout_1)
        dense_2 = tf.keras.layers.BatchNormalization()(dense_2)
        dropout_2 = tf.keras.layers.Dropout(hp.Float('dropout_2', min_value=0.3, max_value=0.6, step=0.1))(dense_2)
        rating_output = tf.keras.layers.Dense(1)(dropout_2)

        user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
        movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
        genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
        rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

        metrics = tfrs.metrics.FactorizedTopK(
            candidates=tf.data.Dataset.from_tensor_slices(self.unique_titles).batch(128).map(movie_model)
        )
        rating_task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()],
        )
        retrieval_task = tfrs.tasks.Retrieval(
            metrics=metrics,
            temperature=0.1
        )
        model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, self.rating_weight, self.retrieval_weight)
        
        # Define the learning rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log'),
            decay_steps=hp.Int('decay_steps', min_value=500, max_value=1000, step=100),
            decay_rate=hp.Float('decay_rate', min_value=0.8, max_value=0.9, step=0.05),
            staircase=True
        )

        # Use legacy Adam on Apple Silicon (Metal backend), standard Adam everywhere else (GCP/Linux/CUDA).
        # tf.keras.optimizers.legacy is not available on Linux. `platform` is imported at the top level.
        if platform.system() == 'Darwin':
            optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=lr_schedule)
        else:
            optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        model.compile(optimizer=optimizer)
        return model


## 5 · Hyperparameter Tuning — Hyperband

**Hyperband** is a bandit-based early-stopping algorithm. It runs many configurations for a small number of epochs, promotes only the best performers to more epochs, and terminates the rest. This finds good hyperparameters far more efficiently than random search or grid search.

**Search space:**

| Hyperparameter | Range |
|---|---|
| `embedding_dimension` | 32 – 256 (step 32) |
| `l2_reg` | 1e-5 – 1e-2 (log scale) |
| `units_1` | 128 – 512 (step 64) |
| `units_2` | 64 – 256 (step 32) |
| `dropout_1 / dropout_2` | 0.3 – 0.6 (step 0.1) |
| `learning_rate` | 1e-4 – 1e-2 (log scale) |
| `decay_steps` | 500 – 1000 (step 100) |
| `decay_rate` | 0.8 – 0.9 (step 0.05) |

Trials are written directly to GCS (`gs://movie-data-1/tuning`) so the search can resume if the VM is interrupted. The objective is `val_root_mean_squared_error` (minimise), matching the upweighted rating task.


In [ ]:
# Objective changed to val_root_mean_squared_error (minimize) to match the
# upweighted rating task (rating_weight=2.0) and address the val RMSE plateau.
timestamp_tuner = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
tuner = Hyperband(
    RecommendationHyperModel(unique_user_ids, unique_titles, len(unique_genres)),
    objective=Objective("val_root_mean_squared_error", direction="min"),
    max_epochs=12,
    factor=3,
    directory='gs://movie-data-1/tuning',   # write directly to GCS
    project_name=f'{timestamp_tuner}/movie_recommendation',
)

tuner.search(
    trainds,
    epochs=12,
    validation_data=testds,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_root_mean_squared_error', patience=3, mode='min')]
)

# Alternatively, you can also reload the tuner and its results later with:
# tuner = Hyperband(
#     RecommendationHyperModel(unique_user_ids, unique_titles, len(unique_genres)),
#     objective=Objective("val_root_mean_squared_error", direction="min"),
#     max_epochs=12,
#     factor=3,
#     directory='gs://movie-data-1/tuning',
#     project_name=f'20260301211834/movie_recommendation',
# )
# tuner.reload()  # loads the previous search results from GCS

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
tuned_model = tuner.hypermodel.build(best_hps)


In [ ]:
# Define callbacks for final training run
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"trained_model/tpe/checkpoints/{timestamp}_cp.ckpt",
    save_best_only=True,
    save_weights_only=True,
    monitor='val_factorized_top_k/top_5_categorical_accuracy', 
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    f"trained_model/tpe/tensorboard/{timestamp}_cp.ckpt",
    histogram_freq=1
)

In [ ]:
# Train the model
history = tuned_model.fit(
    trainds,
    epochs=12,
    validation_data=testds,
    callbacks=[checkpoint_callback, tensorboard_callback]
)

# OR load tuned model weights
# tuned_model.load_weights(f"trained_model/tpe/weights/20260302035534_weights")

In [ ]:
best_hps.values

## 6 · Embedding Extraction — Bridging the Neural Net to XGBoost

The tuned network is used as a **feature extractor**. For every user and every movie, we materialise:

- `user_embedding_dict` — `user_id → float32 vector` (shape `[D]`)
- `movie_embedding_dict` — `title → float32 vector` (shape `[D]`)
- `genre_embedding_dict` — `genre_index → float32 vector` (shape `[D]`)

These are later concatenated into a single feature vector `[user_emb ‖ movie_emb ‖ genre_one_hot]` that XGBoost uses to re-rank candidates. This two-stage design lets the neural net learn rich latent representations while XGBoost handles the precise pairwise ranking objective (`rank:pairwise`).


In [ ]:
def generate_unique_genres(n):
    """
    Generate a one-hot encoded array for `n` unique genres.
    
    Args:
        n (int): Number of unique genres.
    
    Returns:
        np.ndarray: A one-hot encoded array of shape (n, n).
    """
    return np.eye(n, dtype=int).tolist()
    
unique_genres_array = generate_unique_genres(len(unique_genres))

# Extract user and movie embeddings
user_embeddings = tuned_model.user_model.predict(unique_user_ids)
movie_embeddings = tuned_model.movie_model.predict(unique_titles)
genre_embeddings = tuned_model.genre_model.predict(unique_genres_array)


# Create a dictionary to map user IDs and movie IDs to their embeddings
user_embedding_dict = {user_id: embedding for user_id, embedding in zip(unique_user_ids, user_embeddings)}
movie_embedding_dict = {movie_id: embedding for movie_id, embedding in zip(unique_titles, movie_embeddings)}
genre_embedding_dict = {i: embedding for i, embedding in enumerate(genre_embeddings)}

# user_embedding_dict = {k: v for k, v in user_embedding_dict.items()}
movie_embedding_dict = {k.decode('utf-8'): v for k, v in movie_embedding_dict.items()}
# genre_embedding_dict = {k: v for k, v in genre_embedding_dict.items()}

In [ ]:
# print true if Atlas in movie_embedding_dict
print("Atlas" in movie_embedding_dict) if "Atlas" in movie_embedding_dict else print("Atlas not in movie_embedding_dict")

## 7 · XGBoost Ranking Model

XGBoost is trained **only on the neural-net training partition** (`train_combined_dataset`). Using the full dataset here would allow XGBoost to train on interactions that were withheld from the neural network, introducing cross-stage data leakage.

### Feature construction
For every `(user_id, title, genres)` triple in the training set, `create_feature_vector` concatenates:
```
feature = [user_embedding(user_id) ‖ movie_embedding(title) ‖ genres_one_hot]
```

### Cold-start split (mirroring the neural-net split)
XGBoost uses its own 80/20 user-stratified split so validation users are entirely unseen during XGBoost training — the same cold-start discipline applied to the neural network.

### DMatrix group sizes
`rank:pairwise` requires interactions to be grouped by query (user). `dtrain.set_group(group_train)` provides the number of interactions per user so XGBoost constructs correct pairwise training pairs within each group.


In [ ]:

# Build XGBoost dataframe from the TRAINING partition only.
# Using the full combined_dataset here would let XGBoost train on interactions
# that were held out from the neural network, introducing cross-stage leakage.
xgb_df = pd.DataFrame(train_combined_dataset.as_numpy_iterator())
print(xgb_df.head())


In [ ]:
# Create feature vectors by combining user, movie, and genres embeddings
def create_feature_vector(row):
    user_id = row['user_id']
    title = row['title'].decode('utf-8')
    genres = row['genres']
    
    if user_id not in user_embedding_dict:
        raise KeyError(f"User ID {user_id} not found in user_embedding_dict")
    if title not in movie_embedding_dict:
        raise KeyError(f"Title '{title}' not found in movie_embedding_dict")
    
    user_embedding = user_embedding_dict[user_id]
    movie_embedding = movie_embedding_dict[title]
    
    return np.concatenate([user_embedding, movie_embedding, genres])

# Apply the function to create feature vectors
try:
    xgb_df['features'] = xgb_df.apply(create_feature_vector, axis=1)
except KeyError as e:
    print(e)
    

# ── Cold-start XGBoost split ──────────────────────────────────────────────────
# Split by user_id so that val users are entirely unseen during XGBoost training.
# This mirrors the neural net split and gives a realistic evaluation signal.
# No overlap check needed — a user-level split guarantees no row-level overlap.
xgb_all_user_ids = xgb_df['user_id'].unique()
np.random.seed(42)
np.random.shuffle(xgb_all_user_ids)

xgb_split_idx   = int(len(xgb_all_user_ids) * 0.8)
xgb_train_users = set(xgb_all_user_ids[:xgb_split_idx].tolist())
xgb_val_users   = set(xgb_all_user_ids[xgb_split_idx:].tolist())

assert not (xgb_train_users & xgb_val_users), "XGB user leakage detected!"

train_df = xgb_df[xgb_df['user_id'].isin(xgb_train_users)].copy()
val_df   = xgb_df[xgb_df['user_id'].isin(xgb_val_users)].copy()

print(f"XGB train users: {len(xgb_train_users)} ({len(train_df)} interactions)")
print(f"XGB val   users: {len(xgb_val_users)}  ({len(val_df)} interactions)")

X_train = np.vstack(train_df['features'].values)
y_train = train_df['rating'].values
X_val = np.vstack(val_df['features'].values)
y_val = val_df['rating'].values

# Only hstack additional features if they actually exist
remaining_cols = [c for c in train_df.columns if c not in ['user_id', 'title', 'rating', 'genres', 'features']]
if remaining_cols:
    X_train = np.hstack([X_train, train_df[remaining_cols].values])
    X_val   = np.hstack([X_val,   val_df[remaining_cols].values])

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")

# Group sizes for ranking
group_train = train_df.groupby('user_id').size().to_list()
group_val = val_df.groupby('user_id').size().to_list()

# Verify that the sum of group sizes matches the number of rows
assert sum(group_train) == X_train.shape[0], "Mismatch between group sizes and number of rows in X_train"
assert sum(group_val) == X_val.shape[0], "Mismatch between group sizes and number of rows in X_val"

# Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_group(group_train)
dval = xgb.DMatrix(X_val, label=y_val)
dval.set_group(group_val)

In [ ]:
print(xgb_df[['title', 'features']])

In [ ]:
print(train_df.head())

In [ ]:
print(val_df.head())

### 7a · XGBoost Hyperparameter Optimisation — Optuna

**Optuna** runs 25 trials of Bayesian optimisation (Tree-structured Parzen Estimator) to find the best XGBoost hyperparameters.

**Objective:** maximise **per-user NDCG** — the average of `ndcg_score([true_ratings], [pred_ratings])` computed independently for every validation user who has more than one interaction. This is the correct approach; computing NDCG over the entire validation set as a single query trivially inflates the score to ~0.99.

Predicted scores are min-max scaled to `[0.5, 5.0]` before ranking so that NDCG operates on a meaningful absolute scale.

**Search space:** `eta`, `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `gamma`, `lambda`.


In [ ]:
# Define the objective function for Optuna
def objective(trial):
    param = {
        'objective': 'rank:pairwise',
        'eval_metric': 'ndcg',
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'lambda': trial.suggest_float('lambda', 0.0, 1.0),
    }
    
    bst = xgb.train(param, dtrain, num_boost_round=100, evals=[(dval, 'eval')],
                    early_stopping_rounds=10, verbose_eval=False)
    y_pred = bst.predict(dval)

    min_pred, max_pred = np.min(y_pred), np.max(y_pred)
    y_pred_scaled = 0.5 + (y_pred - min_pred) * 4.5 / (max_pred - min_pred)

    # ✅ Per-user NDCG — same approach as your evaluation cell
    user_ndcg_scores = []
    for uid in val_df['user_id'].unique():
        mask = val_df['user_id'] == uid
        true_r = val_df.loc[mask, 'rating'].values
        pred_r = y_pred_scaled[mask]
        if len(true_r) > 1:
            user_ndcg_scores.append(ndcg_score([true_r], [pred_r]))

    return np.mean(user_ndcg_scores)

# Create a study and optimize the objective function
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=25)

# Get the best parameters
best_params = study.best_params

print(f"Best parameters: {best_params}")


In [ ]:
bst = xgb.train(
    best_params,
    dtrain,
    num_boost_round=500,
    evals=[(dval, 'eval')],
    early_stopping_rounds=20,
    verbose_eval=10,
    custom_metric=None,
    evals_result={},
    # min_delta is not a native XGBoost param, so use callbacks instead
    callbacks=[
        xgb.callback.EarlyStopping(
            rounds=20,
            metric_name='rmse',
            data_name='eval',
            min_delta=1e-4  # stop if improvement < 0.0001
        )
    ]
)

# OR Reload previously trained bst model from file
# bst = xgb.Booster()
# bst.load_model("trained_model/xgb/20260302035556_xgb_model.json")

print(f"Best iteration: {bst.best_iteration}, Best RMSE: {bst.best_score:.5f}")



In [ ]:
# Save best_params for xgboost and best_hps for keras tuner
np.save('best_params_xgb.npy', best_params)
np.save('best_hps.npy', best_hps.values)

In [ ]:
# print length unique val df titles
decoded_titles = [t.decode('utf-8') if isinstance(t, bytes) else t for t in val_df['title']]
print(f"Unique titles in val_df: {len(np.unique(decoded_titles))}")

## 8 · Inference Pipeline

The inference pipeline is a two-stage cascade:

```
XGBoost ranking
  rank_movies_with_xgb(user_id, movies_df, bst, k=K*100)
        │  retrieves top-K*100 candidates
        ▼
Hypermodel re-scoring
  rate_movies_with_hypermodel(hypermodel, user_id, titles, genres)
        │  predicts a precise rating for each candidate
        ▼
get_final_predictions → top-K movies sorted by predicted rating
```

- **`get_user_titles_rated`** — fetches all titles a user has already rated so they can be filtered from candidates
- **`create_rank_feature_vector`** — constructs the XGBoost feature vector for a single (user, movie) pair; uses `np.zeros_like` of an existing embedding as a fallback for out-of-vocabulary titles
- **`rank_movies_with_xgb`** — scores all candidate movies with XGBoost and min-max scales scores to `[0.5, 5.0]`; supports both strict filtering (`remove_all_rated=True`) and soft filtering (`remove_all_rated=False`)


In [ ]:
# Get titles a user already rated
def get_user_titles_rated(user_id):
    rated = ratings_bq[ratings_bq['user_id'] == user_id]['title'].values
    if len(rated) == 0:
        logger.warning(f"No ratings found for user {user_id}. Returning empty array.")
    return rated

In [ ]:
movies_df = pd.DataFrame(movies.as_numpy_iterator())

In [ ]:
def create_rank_feature_vector(user_id, title, genres):
    if user_id not in user_embedding_dict:
        raise KeyError(f"User ID {user_id} not found in user_embedding_dict")
    user_embedding = user_embedding_dict[user_id]

    title = title.decode('utf-8') if isinstance(title, bytes) else title
    if title not in movie_embedding_dict:
        # Derive size from an existing embedding rather than hardcoding
        movie_embedding = np.zeros_like(next(iter(movie_embedding_dict.values())))
    else:
        movie_embedding = movie_embedding_dict[title]
    
    # Combine user embedding, movie embedding, and genres
    return np.concatenate([user_embedding, movie_embedding, genres])

In [ ]:
def rank_movies_with_xgb(user_id, xgb_combined_dataset, bst, k=5, remove_all_rated=True):
    if user_id not in user_embedding_dict:
        raise ValueError(f"User ID {user_id} not found in user_embedding_dict")
    already_rated = get_user_titles_rated(user_id)
    print(f"User {user_id} has rated {len(already_rated)} movies.")
    
    candidate_features = []
    for _, row in xgb_combined_dataset.iterrows():
        title = row['title']
        genres = row['genres']
        # Only use columns that are not title, genres, or any target label
        additional_features = row.drop(labels=[c for c in ['title', 'genres'] if c in row.index]).values
        feature_vector = create_rank_feature_vector(user_id, title, genres)
        candidate_features.append(np.concatenate([feature_vector, additional_features]))
    
    candidate_features = np.vstack(candidate_features)
    dtest = xgb.DMatrix(candidate_features)
    predicted_ratings = bst.predict(dtest)

    min_rating, max_rating = 0.5, 5.0
    min_pred = np.min(predicted_ratings)
    max_pred = np.max(predicted_ratings)
    predicted_ratings_scaled = min_rating + (predicted_ratings - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)

    # Decode titles once here to avoid double-decode in get_final_predictions
    titles = [t.decode('utf-8') if isinstance(t, bytes) else t for t in xgb_combined_dataset['title'].values]
    ranked_movies = list(zip(predicted_ratings_scaled, titles))
    print("Ranked Movies Before Filtering:", ranked_movies[:10])

    if remove_all_rated:
        ranked_movies = [movie for movie in ranked_movies if movie[1] not in already_rated]
        ranked_movies.sort(reverse=True, key=lambda x: x[0])
        return ranked_movies[:k]
    else:
        random.shuffle(ranked_movies)
        filtered_movies = []
        already_rated_count = 0
        for movie in ranked_movies:
            if movie[1] in already_rated:
                already_rated_count += 1
                # Keep only 3 out of every 10 already-rated movies
                if already_rated_count % 10 >= 3:
                    continue
            filtered_movies.append(movie)
        filtered_movies.sort(reverse=True, key=lambda x: x[0])
        return filtered_movies[:k]

In [ ]:
# Example usage
user_id = 163298


top_k_movies = rank_movies_with_xgb(user_id, movies_df, bst, k=5, remove_all_rated=True)
print(f'Ranked movies for user {user_id}: {top_k_movies}')

## 8b · XGBoost Evaluation — Per-User Metrics

Evaluate the trained XGBoost ranker on the held-out validation users using three complementary metrics:

- **NDCG (Normalised Discounted Cumulative Gain)** — rewards ranking truly liked movies higher; averaged across all validation users with more than one interaction. Computed per-user (not globally) to avoid trivial inflation.
- **Precision@10** — fraction of the top-10 predicted movies that are actually liked (≥ threshold)
- **Recall@10** — fraction of liked movies that appear in the top-10 predictions
- **Accuracy@10** — fraction of users for whom at least one liked movie appears in the top-10

All three are evaluated at thresholds `3.5`, `4.0`, and `4.5` to measure sensitivity to the definition of "liked". Users with no relevant items at a given threshold are excluded to avoid dividing by zero.


In [ ]:

# Predict ratings for the validation set
y_pred = bst.predict(dval)
min_rating = 0.5
max_rating = 5.0
min_pred = np.min(y_pred)
max_pred = np.max(y_pred)
y_pred_scaled = min_rating + (y_pred - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)

# Assign predicted ratings and rank per user
# NOTE: these two lines were previously commented out, which caused Precision/Recall/Accuracy@K
# to reference an undefined (or stale) `rank` column and produced misleading results.
val_df = val_df.copy()
val_df['predicted_rating'] = y_pred_scaled
val_df['rank'] = val_df.groupby('user_id')['predicted_rating'].rank(ascending=False, method='first')

# ── Per-user NDCG (correct) ───────────────────────────────────────────────────
# Global ndcg_score([y_val], [y_pred_scaled]) treats the entire validation set
# as one query, trivially inflating the score (~0.988). The correct approach is
# to compute NDCG independently for each user and then average.
user_ndcg_scores = []
for uid, user_data in val_df.groupby('user_id'):  # renamed to avoid shadowing outer user_id
    true_ratings = user_data['rating'].values
    pred_ratings = user_data['predicted_rating'].values
    if len(true_ratings) > 1:
        user_ndcg_scores.append(ndcg_score([true_ratings], [pred_ratings]))

ndcg = np.mean(user_ndcg_scores)
print(f'NDCG Score (per-user average over {len(user_ndcg_scores)} users): {ndcg:.4f}')

# ── Precision / Recall / Accuracy @K ─────────────────────────────────────────
threshold = 4.0  # only genuinely liked movies

def get_relevant_items_strict(user_data, threshold=4.0):
    relevant = user_data[user_data['rating'] >= threshold]['title'].values
    return relevant  # no fallback — if empty, skip this user

def precision_at_k_strict(val_df, k, threshold=4.0):
    precision_scores = []
    for uid in val_df['user_id'].unique():
        user_data    = val_df[val_df['user_id'] == uid]
        true_titles  = get_relevant_items_strict(user_data, threshold)
        if len(true_titles) == 0:
            continue  # skip users with no relevant items at this threshold
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        denom = min(k, len(user_data))
        precision_scores.append(len(set(true_titles).intersection(top_k_titles)) / denom)
    return np.mean(precision_scores)

def recall_at_k_strict(val_df, k, threshold=4.0):
    recall_scores = []
    for uid in val_df['user_id'].unique():
        user_data   = val_df[val_df['user_id'] == uid]
        true_titles = get_relevant_items_strict(user_data, threshold)
        if len(true_titles) == 0:
            continue
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        recall_scores.append(len(set(true_titles).intersection(top_k_titles)) / len(true_titles))
    return np.mean(recall_scores)

def accuracy_at_k_strict(val_df, k, threshold=4.0):
    correct_count = 0
    eligible_users = 0
    for uid in val_df['user_id'].unique():
        user_data   = val_df[val_df['user_id'] == uid]
        true_titles = get_relevant_items_strict(user_data, threshold)
        if len(true_titles) == 0:
            continue
        eligible_users += 1
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        if set(true_titles).intersection(top_k_titles):
            correct_count += 1
    return correct_count / eligible_users if eligible_users > 0 else 0.0

for threshold in [3.5, 4.0, 4.5]:
    p = precision_at_k_strict(val_df, 10, threshold)
    r = recall_at_k_strict(val_df, 10, threshold)
    a = accuracy_at_k_strict(val_df, 10, threshold)
    print(f"Threshold {threshold}: Precision@10={p:.4f} | Recall@10={r:.4f} | Accuracy@10={a:.4f}")

In [ ]:

# Additional diagnostics
print("Distribution of Ratings in Training Set:")
print(train_df['rating'].value_counts())

print("Distribution of Ratings in Validation Set:")
print(val_df['rating'].value_counts())

print("Predictions Distribution:")
print(pd.Series(y_pred_scaled).describe())

## 9 · Hypermodel Rating Predictions

`rate_movies_with_hypermodel` runs the tuned two-tower model in inference mode to produce a precise predicted rating for each `(user, title, genre)` triple. This is the second stage of the cascade — it re-scores the top candidates returned by XGBoost using the richer neural representation.

`get_final_predictions` wires both stages together:
1. Retrieves `k × 100` candidates via XGBoost (wide retrieval net)
2. Passes them through the hypermodel for precise rating prediction
3. Returns the top `k` movies sorted by predicted rating


In [ ]:
# print genre for given movieId using movies_bq
def get_genre(title):
    result = movies_bq[movies_bq['title'] == title]['genres'].values
    if len(result) == 0:
        raise ValueError(f"Title '{title}' not found in movies_bq")
    return result[0]
        
def rate_movies_with_hypermodel(hypermodel, user_id, titles, genres):
    # Use the recommendation hypermodel to predict ratings for the given user and movie titles
    predicted_ratings = []
    for title, genre in zip(titles, genres):
        # Predict ratings using the hypermodel
        _, _, _, predicted_rating = hypermodel({
            "user_id": np.array([user_id]),
            "title": np.array([title]),
            "genres": np.array([genre])
        })
        predicted_ratings.append([title, predicted_rating.numpy()[0][0]])
    return predicted_ratings

# Re-assert user_id in case it was shadowed upstream
user_id = 163298
movie_titles = ['The Rescue', 'Siberian Sniper']
movie_genres = [get_genre(title) for title in movie_titles]
predicted_ratings = rate_movies_with_hypermodel(tuned_model, user_id, movie_titles, movie_genres)
print("Predicted ratings:")
print(predicted_ratings)

In [ ]:
def get_final_predictions(user_id, combined_dataset, bst, hypermodel, k=5, remove_all_rated=True):
    # Rank movies using the XGBoost model
    num_retrieval = k * 100
    top_k_movies = rank_movies_with_xgb(user_id, combined_dataset, bst, k=num_retrieval, remove_all_rated=remove_all_rated)

    # Guard against empty candidate list
    if not top_k_movies:
        raise ValueError(f"No candidate movies returned for user {user_id}. "
                         "The user may have rated all available movies.")

    # Get the movie titles and genres
    # titles are already decoded strings — rank_movies_with_xgb handles decoding
    _, movie_titles = zip(*top_k_movies)

    movie_genres = [get_genre(title) for title in movie_titles]

    # Predict ratings using the hypermodel
    final_predictions = rate_movies_with_hypermodel(hypermodel, user_id, movie_titles, movie_genres)

    # Sort the final predictions by rating
    final_predictions = sorted(final_predictions, key=lambda x: x[1], reverse=True)

    return final_predictions[:k]

# Re-assert user_id in case it was shadowed or stale from upstream cells
user_id = 163298
final_predictions = get_final_predictions(user_id, movies_df, bst, tuned_model, k=3, remove_all_rated=True)
print("Final Predictions:")
print(final_predictions)

## 10 · Neural Network Evaluation

Run `model.evaluate` over the full test dataset to obtain:

| Metric | What it measures |
|---|---|
| `root_mean_squared_error` | Rating prediction accuracy |
| `factorized_top_k/top_1_categorical_accuracy` | Retrieval: correct movie in top-1 |
| `factorized_top_k/top_5_categorical_accuracy` | Retrieval: correct movie in top-5 |
| `factorized_top_k/top_10_categorical_accuracy` | Retrieval: correct movie in top-10 |
| `factorized_top_k/top_50_categorical_accuracy` | Retrieval: correct movie in top-50 |
| `factorized_top_k/top_100_categorical_accuracy` | Retrieval: correct movie in top-100 |

> No `steps` argument is passed — `testds` is a finite dataset; Keras stops automatically at the last batch, including any partial final batch that integer division would otherwise drop.


In [ ]:
# Evaluate on the full test set — no steps argument needed.
# testds is a finite dataset; Keras will automatically stop at the end,
# including the last partial batch that integer division would have dropped.
metrics = tuned_model.evaluate(testds, return_dict=True)

In [ ]:
print(f"Retrieval top-1 accuracy: {metrics['factorized_top_k/top_1_categorical_accuracy']:.3f}.")
print(f"Retrieval top-5 accuracy: {metrics['factorized_top_k/top_5_categorical_accuracy']:.3f}.")
print(f"Retrieval top-10 accuracy: {metrics['factorized_top_k/top_10_categorical_accuracy']:.3f}.")
print(f"Retrieval top-50 accuracy: {metrics['factorized_top_k/top_50_categorical_accuracy']:.3f}.")
print(f"Retrieval top-100 accuracy: {metrics['factorized_top_k/top_100_categorical_accuracy']:.3f}.")
print(f"Ranking RMSE: {metrics['root_mean_squared_error']:.3f}.")

## 11 · Saving Artifacts

Three artifact types are saved locally (and can be synced to GCS):

| Artifact | Format | Location |
|---|---|---|
| Keras model weights | TF SavedModel (no `.h5`) | `trained_model/tpe/weights/<timestamp>_weights/` |
| XGBoost model | JSON | `trained_model/xgb/<timestamp>_xgb_model.json` |
| FAISS index | Binary | `trained_model/faiss/<timestamp>_faiss.index` |

Every save cell uses a **fresh `datetime` timestamp** (not reused from earlier cells) and calls `os.makedirs(..., exist_ok=True)` so runs never fail due to a missing directory.


In [ ]:
# Ensure the save directory exists
save_timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
weights_dir = f"trained_model/tpe/weights/{save_timestamp}_weights"
os.makedirs(weights_dir, exist_ok=True)

# Save in TF SavedModel format (recommended over .h5 with legacy Keras)
tuned_model.save_weights(weights_dir)

In [ ]:

# tuned_model.save() fails with FailedPreconditionError because FactorizedTopK
# embeds a tf.data.Dataset pipeline (stateful ResourceGather ops) in the graph,
# which cannot be serialized to SavedModel format.
#
# Instead, save each inference sub-model individually — none of them hold a
# reference to the candidates dataset, so they serialize cleanly.
sub_models = {
    "user_model":   tuned_model.user_model,
    "movie_model":  tuned_model.movie_model,
    "genre_model":  tuned_model.genre_model,
    "rating_model": tuned_model.rating_model,
}
for name, sub_model in sub_models.items():
    sub_model_path = os.path.join(weights_dir, name)
    sub_model.save(sub_model_path)
    print(f"Saved {name} → {sub_model_path}")


In [ ]:
# Save xgb model — use a fresh timestamp and ensure the directory exists.
xgb_save_timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
xgb_save_dir = "trained_model/xgb"
os.makedirs(xgb_save_dir, exist_ok=True)

bst.save_model(f"{xgb_save_dir}/{xgb_save_timestamp}_xgb_model.json")

## 12 · FAISS Retrieval Index

FAISS (`IndexFlatIP`) enables sub-millisecond exact nearest-neighbour (ANN) search over the full movie catalogue. At inference time, a user's embedding is used to query the index and retrieve the top-K most similar movies by **inner product (dot-product) similarity** — a fast first-stage retrieval that replaces brute-force scoring.

### Why `IndexFlatIP` and not `IndexFlatL2`?

TFRS's `Retrieval` task trains the user and movie towers by maximising **dot-product similarity** between matched pairs (via a sampled softmax loss). Because the model was optimised for inner product, querying FAISS with the same metric produces rankings that are consistent with the training objective.

`IndexFlatL2` measures **Euclidean distance**, which penalises embeddings that point in the same direction but differ in magnitude. Without explicit L2 normalisation of the embedding layers, `IndexFlatL2` and `IndexFlatIP` diverge and L2 would retrieve suboptimal candidates — movies that are geometrically close but not semantically aligned with the user.

| Index | Use when |
|---|---|
| `IndexFlatIP` | Embeddings trained with dot-product / inner-product loss (TFRS `Retrieval`, matrix factorisation) ✅ |
| `IndexFlatL2` | Embeddings trained with Euclidean/contrastive loss, or explicitly L2-normalised before indexing |

> **Optional hardening:** calling `faiss.normalize_L2(embeddings)` on both the movie embeddings (at index build time) and the user query vector (at search time) converts inner product into **cosine similarity**. This removes magnitude variance and is common practice with two-tower models, but is not strictly required here since the towers are not normalised during training.

### Key implementation details
- **`extract_embeddings`** — batched extraction (default `batch_size=512`) avoids OOM on large catalogues; output is cast to `float32`
- **`index_movie_embeddings`** — validates shape, ensures C-contiguous `float32` layout (both required by FAISS), then adds all embeddings to `IndexFlatIP` in one call
- **`recommend_movies`** — clamps `k = min(k, index.ntotal)` so the call never fails when `k` exceeds the catalogue size; returns decoded string titles; higher IP score = more similar (opposite sign convention to L2 distance)


In [ ]:
trained_user_embeddings, trained_movie_embeddings, trained_genre_embeddings, predicted_rating = tuned_model({
    "user_id": np.array([163298]),
    "title": np.array(['The Banshees of Inisherin']),
    # Cast to float32 to match the genre_input layer dtype
    "genres": np.array([get_genre('The Banshees of Inisherin')], dtype=np.float32)
})
print("Predicted rating:")
print(predicted_rating)

In [ ]:
def extract_embeddings(model, unique_user_ids, unique_titles, batch_size=512):
    """
    Extract embeddings for all users and movies using the trained model.
    
    Args:
    - model: The trained recommendation model.
    - unique_user_ids: List of unique user IDs.
    - unique_titles: List of unique movie titles.
    - batch_size: Number of items to process per batch.
    
    Returns:
    - user_embeddings: Embeddings for all users.
    - movie_embeddings: Embeddings for all movies.
    """
    # Extract movie embeddings in batches
    titles = np.array(unique_titles)
    movie_embeddings = np.vstack([
        model.movie_model(tf.constant(titles[i:i+batch_size], dtype=tf.string)).numpy()
        for i in range(0, len(titles), batch_size)
    ]).astype(np.float32)  # cast to float32 for FAISS compatibility

    # Extract user embeddings in batches
    user_ids = np.array(unique_user_ids)
    user_embeddings = np.vstack([
        model.user_model(tf.constant(user_ids[i:i+batch_size], dtype=tf.int32)).numpy()
        for i in range(0, len(user_ids), batch_size)
    ]).astype(np.float32)

    return user_embeddings, movie_embeddings

In [ ]:
def index_movie_embeddings(movie_embeddings):
    """
    Index the movie embeddings using FAISS.
    
    Args:
    - movie_embeddings: Embeddings for all movies (np.ndarray, float32).
    
    Returns:
    - index: FAISS index with movie embeddings.
    """
    if movie_embeddings.ndim != 2 or movie_embeddings.shape[0] == 0:
        raise ValueError(f"Expected a non-empty 2D array, got shape {movie_embeddings.shape}")

    # Ensure float32 and C-contiguous layout — both required by FAISS
    movie_embeddings = np.ascontiguousarray(movie_embeddings, dtype=np.float32)

    embedding_dimension = movie_embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dimension)
    index.add(movie_embeddings)

    return index

In [ ]:
def recommend_movies(model, index, unique_titles, user_id, k=10):
    """
    Recommend movies for a given user by querying the FAISS index.
    
    Args:
    - model: The trained recommendation model.
    - index: FAISS index with movie embeddings.
    - unique_titles: List of unique movie titles.
    - user_id: The user ID for which to make recommendations.
    - k: Number of recommendations to retrieve (default is 10).
    
    Returns:
    - recommended_movies: List of recommended movie titles.
    """
    # Guard against k exceeding the number of indexed movies
    k = min(k, index.ntotal)
    if k == 0:
        raise ValueError("FAISS index is empty — no movies to recommend.")

    # Get the embedding for the given user, cast to float32 for FAISS
    user_embedding = np.ascontiguousarray(
        model.user_model(tf.constant([user_id], dtype=tf.int32)).numpy(),
        dtype=np.float32
    )

    # Query the FAISS index
    distances, indices = index.search(user_embedding, k)

    # Get the recommended movie titles, decoded to strings
    recommended_movies = [
        t.decode('utf-8') if isinstance(t, bytes) else t
        for t in np.array(unique_titles)[indices[0]]
    ]

    return recommended_movies

In [ ]:
# Extract embeddings
user_embeddings, movie_embeddings = extract_embeddings(tuned_model, unique_user_ids, unique_titles)

# Index movie embeddings
index = index_movie_embeddings(movie_embeddings)

# Example user ID for which to make recommendations
example_user_id = 163298

recommended_movie_ids = recommend_movies(tuned_model, index, unique_titles, example_user_id, k=5)

print(f"Recommended movie titles for user {example_user_id}: {recommended_movie_ids}")

In [ ]:
# Ensure the FAISS save directory exists before writing the index
faiss_save_dir = "trained_model/faiss"
os.makedirs(faiss_save_dir, exist_ok=True)
faiss.write_index(index, f"{faiss_save_dir}/{timestamp}_faiss.index")